In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._load_vn30_binary import preprocess, VN30, TARGETS
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

In [3]:
X_train, y_train = preprocess("ACB", verbose=True)["train"]

=== Preprocessing ACB ===
Train: (1213, 124) | Valid: None | Test: (328, 124)
Label dist train: Counter({0: 643, 1: 570}), test: Counter({0: 175, 1: 153})


In [4]:
acc = []
for symbol in VN30:
    data = preprocess(symbol, lag=30)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]

    tscv = TimeSeriesSplit(n_splits=5)

    param_dist = {
        "n_estimators": [10, 20, 50, 100],
        "max_depth": [3, 5, 7, 9],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(),
        param_distributions=param_dist,
        n_iter=10,        
        cv=tscv,             
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, Y_train)

    train_preds = search.predict(X_train)
    test_preds = search.predict(X_test)

    print(f"Symbol: {symbol}")
    print(f"Train Balanced Accuracy: {balanced_accuracy_score(Y_train, train_preds)}")
    print(f"Test Balanced Accuracy: {balanced_accuracy_score(Y_test, test_preds)}")
    print(f"Test Confusion Matrix:\n{confusion_matrix(Y_test, test_preds)}\n")

    acc.append(balanced_accuracy_score(Y_test, test_preds))

# Mean of best 10
mean_acc = np.mean(acc)
print(f"Mean Test Balanced Accuracy (Best 10): {mean_acc}")

Symbol: ACB
Train Balanced Accuracy: 0.7660077487653816
Test Balanced Accuracy: 0.5225583566760037
Test Confusion Matrix:
[[136  39]
 [112  41]]

Symbol: BCM
Train Balanced Accuracy: 0.5504201680672269
Test Balanced Accuracy: 0.5052907042894527
Test Confusion Matrix:
[[185   2]
 [138   3]]

Symbol: BID
Train Balanced Accuracy: 0.6154432762201454
Test Balanced Accuracy: 0.5303244544655482
Test Confusion Matrix:
[[151  40]
 [100  37]]

Symbol: BVH
Train Balanced Accuracy: 0.602999543299552
Test Balanced Accuracy: 0.5103194103194103
Test Confusion Matrix:
[[172  13]
 [130  13]]

Symbol: CTG
Train Balanced Accuracy: 0.6750630437779237
Test Balanced Accuracy: 0.4988819320214669
Test Confusion Matrix:
[[123  33]
 [136  36]]

Symbol: FPT
Train Balanced Accuracy: 0.8995469255663431
Test Balanced Accuracy: 0.48141118678128836
Test Confusion Matrix:
[[75 84]
 [86 83]]

Symbol: GAS
Train Balanced Accuracy: 0.6753867035097348
Test Balanced Accuracy: 0.49116518773976053
Test Confusion Matrix:
[[186